In [2]:
import torch
from torch import nn
from torch.nn import functional as F
import math

# TransUNet — CNN 抓细节 + Transformer 看全局

## 和 U-Net 的关系

U-Net 的问题：Bottleneck 只用了两个 3×3 卷积，感受野太小，最深层看不到全图的语义关系。

TransUNet 的解法：把 U-Net 的 Bottleneck（DoubleConv）换成 12 层 Transformer Encoder。

| | U-Net | TransUNet |
|---|---|---|
| Encoder | 4 级 CNN Down | 3 级 CNN Down（浅层细节）|
| Bottleneck | DoubleConv(512→1024) | Patch Embedding + 12 层 Transformer |
| Decoder | 4 级上采样 + skip | 3 级上采样 + skip |
| Skip 来源 | Encoder 的 CNN 特征 | Encoder 的 CNN 特征 |

## 数据流

```
输入 (B, 3, H, W)
  ↓
[CNN Encoder — 3 级下采样，提取局部细节]
  enc1 (B, 64,  H,   W)   ────────────────────→ Dec1
  enc2 (B, 128, H/2, W/2)  ──────────→ Dec2
  enc3 (B, 256, H/4, W/4)  ─→ Dec3
  ↓
[Patch Embedding — CNN 特征图 → Transformer token]
  (B, 256, H/8, W/8) → Conv1x1 → (B, N, d_model)
  + Position Embedding
  ↓
[Transformer Encoder × 12 — 全局自注意力]
  每个 token 能看到所有其他 token
  ↓
[Reshape — token 序列 → 特征图]
  (B, N, d_model) → (B, d_model, H/8, W/8)
  → Conv → (B, 256, H/8, W/8)  ← 投影回 CNN 通道
  ↓
[CNN Decoder — 3 级上采样 + 跳跃连接]
  Dec3 ← skip enc3
  Dec2 ← skip enc2
  Dec1 ← skip enc1
  ↓
1×1 Conv → (B, n_classes, H, W)
```

**核心思想**：CNN 负责局部细节（contour、边界），Transformer 负责全局关系（器官之间的空间位置关系），两个各干各的，拼起来。

## 1. CNN 组件 — DoubleConv / Down / Up

和 U-Net 基本一样。唯一的改动：Up 模块多了一个 `encoder_channels` 参数。

**为什么**：U-Net 里 encoder 和 upsampled decoder 输出通道数天然相等（对称设计），concat 后正好是 `in_channels`。TransUNet 的 Transformer 打乱了这条规则——上采样路径的通道数和 Encoder 特征图通道数不再相等。

In [3]:
class DoubleConv(nn.Module):
    """两次 3×3 卷积 + BN + ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

In [4]:
class Down(nn.Module):
    """CNN Encoder 的一级：DoubleConv → MaxPool 下采样"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        before_pool = self.conv(x)    # 存下来给跳跃连接用
        after_pool = self.pool(before_pool)
        return before_pool, after_pool

In [5]:
class Up(nn.Module):
    """Decoder 的一级：转置卷积上采样 → concat Encoder 特征 → DoubleConv
    
    Args:
        in_channels:      上一层（更深层）传来的通道数
        out_channels:     输出通道数
        encoder_channels: 跳跃连接来的 Encoder 特征通道数（默认 = out_channels，U-Net 兼容）
    
    U-Net 用法（对称）：Up(1024, 512) → encoder=512, up:1024→512, concat:512+512=1024 ✓
    TransUNet 用法：    Up(256, 128, encoder_channels=256) → up:256→128, concat:256+128=384 ✓
    """
    def __init__(self, in_channels, out_channels, encoder_channels=None):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = out_channels   # U-Net 对称默认
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        # concat 后的总通道数 = encoder 通道 + 上采样后的通道
        self.conv = DoubleConv(encoder_channels + out_channels, out_channels)

    def forward(self, x_encoder, x_decoder):
        x_up = self.up(x_decoder)

        # 尺寸对齐（处理奇数输入）
        diff_h = x_encoder.size(2) - x_up.size(2)
        diff_w = x_encoder.size(3) - x_up.size(3)
        x_up = F.pad(x_up, [
            diff_w // 2, diff_w - diff_w // 2,
            diff_h // 2, diff_h - diff_h // 2,
        ])

        # 跳跃连接：Encoder 浅层特征 ⊕ Decoder 深层特征
        x = torch.cat([x_encoder, x_up], dim=1)
        return self.conv(x)

## 2. Transformer 组件 — 复用 ViT 的 SelfAttention / MultiHeadAttention / FeedForward

和 ViT 完全一样，唯一区别：**没有 CLS token**（不分类，做 dense 预测）。

In [6]:
class SelfAttention(nn.Module):
    """单头自注意力"""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        attn = self.softmax(scores)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        return out, attn

In [7]:
class MultiHeadAttention(nn.Module):
    """多头注意力：Pre-LN 版本，不做残差和 Norm"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.self_attn = SelfAttention(dropout)
        self.fc = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v):
        batch_size = q.size(0)
        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn = self.self_attn(Q, K, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        out = self.fc(out)
        out = self.dropout(out)
        return out, attn

In [8]:
class FeedForward(nn.Module):
    """MLP：升维 → GELU → 降维"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

In [9]:
class TransformerEncoder(nn.Module):
    """Pre-LN Transformer Encoder（和 ViT 完全一样）"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src):
        # Pre-LN: Norm → Sublayer → Add
        out = src + self.attn(self.norm1(src), self.norm1(src), self.norm1(src))[0]
        out = out + self.ffn(self.norm2(out))
        return out

## 3. PatchEmbed — CNN 特征图 → Transformer Token

和 ViT 切原图不同，这里切的是 **CNN 已经抽象过的特征图**（256 通道，不是 3 通道 RGB）。

`patch_size=1` 意味着**每个空间位置就是一个 token**——CNN 的局部融合能力已经帮你把邻域信息编码进去了，不需要再切大块。

In [ ]:
class PatchEmbed(nn.Module):
    """CNN 特征图 → 序列 token
    
    输入：(B, C, H, W) — CNN Encoder 最后一层输出的特征图
    输出：(B, N, d_model) — N 个 token，每个 d_model 维
    """
    def __init__(self, img_size, patch_size, in_channels,  d_model):
        super().__init__()
        assert img_size % patch_size == 0, \
            f"img_size ({img_size}) must be divisible by patch_size ({patch_size})"
        self.n_patches = (img_size // patch_size) ** 2
        # Conv2d 做 projection：C → d_model
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)            # (B, d_model, H/P, W/P)
        x = x.flatten(2)            # (B, d_model, N)
        return x.transpose(1, 2)    # (B, N, d_model)

## 4. TransUNet 完整模型

CNN Encoder（3级）→ Transformer Bottleneck（12层）→ CNN Decoder（3级 + skip）。

**和 ViT 的区别**：
1. 没有 CLS token — 不是分类，每个 token 对应一个空间位置
2. 位置编码没有用 ViT 的 B.4 节截断正态初始化 — 用的是标准零均值截断正态

In [ ]:
class TransUNet(nn.Module):
    """TransUNet：CNN Encoder + Transformer Bottleneck + CNN Decoder
    
    Args:
        in_channels:  输入图像通道数
        n_classes:    分割类别数
        img_size:     输入图像尺寸（假设正方形）
        cnn_features: CNN Encoder 每层通道 [enc1, enc2, enc3]
        d_model:      Transformer 隐层维度
        n_heads:      多头注意力头数
        d_ff:         FFN 中间层维度
        num_layers:   Transformer Encoder 层数
        patch_size:   token 化的 patch 大小（通常 1）
    """
    def __init__(self, in_channels=3, n_classes=1, img_size=224,
                 cnn_features=[64, 128, 256],
                 d_model=512, n_heads=8, d_ff=2048, num_layers=12,
                 patch_size=16, dropout=0.1):
        super().__init__()

        # ========================
        # CNN Encoder（3 级下采样，对应 U-Net 的前半截）
        # ========================
        self.enc1 = Down(in_channels, cnn_features[0])      # → (B, 64,  H,   W)
        self.enc2 = Down(cnn_features[0], cnn_features[1])  # → (B, 128, H/2, W/2)
        self.enc3 = Down(cnn_features[1], cnn_features[2])  # → (B, 256, H/4, W/4)

        # ========================
        # Transformer Bottleneck
        # ========================
        img_size_after_cnn = img_size // 8   # 3 次 MaxPool → H/8
        self.patch_embed = PatchEmbed(
            in_channels=cnn_features[2],     # CNN 输出的通道数
            img_size=img_size_after_cnn,     # H/8 × W/8
            patch_size=patch_size,
            d_model=d_model,
        )
        n_patches = self.patch_embed.n_patches

        # 位置编码（可学习）
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, d_model))

        # 12 层 Transformer Encoder
        self.transformer_layers = nn.ModuleList([
            TransformerEncoder(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # ========================
        # Decoder 投影层
        # ========================
        # Transformer 输出 (B, N, d_model)
        # → reshape 回空间格式 → 投影回 CNN 通道
        self.decoder_proj = nn.Sequential(
            nn.Conv2d(d_model, cnn_features[2], kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_features[2]),
            nn.ReLU(inplace=True),
        )

        # ========================
        # CNN Decoder（3 级上采样 + 跳跃连接）
        # ========================
        # dec3：喂入的是 256 通道（encoder 出来的深层特征），
        #       跳跃连接 enc3 也是 256 通道（encoder 浅层特征）
        #       concat = 256+128 = 384 → DoubleConv 压到 128
        self.dec3 = Up(cnn_features[2], cnn_features[1],
                       encoder_channels=cnn_features[2])   # → (B, 128, H/4, W/4)
        
        # dec2：上一层传下来 128 通道，enc2 也是 128 通道
        #       concat = 128+64 = 192 → DoubleConv 压到 64
        self.dec2 = Up(cnn_features[1], cnn_features[0],
                       encoder_channels=cnn_features[1])   # → (B, 64, H/2, W/2)
        
        # dec1：上一层传下来 64 通道，enc1 是 64 通道
        #       concat = 64+64 = 128 → DoubleConv 压到 64
        self.dec1 = Up(cnn_features[0], cnn_features[0],
                       encoder_channels=cnn_features[0])   # → (B, 64, H, W)

        # ========================
        # Head：1×1 卷积，逐像素分类
        # ========================
        self.head = nn.Conv2d(cnn_features[0], n_classes, kernel_size=1)

        # ========================
        # 权重初始化
        # ========================
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weight)

    def _init_weight(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        B = x.size(0)
        H, W = x.shape[2], x.shape[3]

        # ---------- ① CNN Encoder ----------
        enc1, x = self.enc1(x)   # enc1: (B, 64,  H,   W), x: (B, 64,  H/2, W/2)
        enc2, x = self.enc2(x)   # enc2: (B, 128, H/2, W/2), x: (B, 128, H/4, W/4)
        enc3, x = self.enc3(x)   # enc3: (B, 256, H/4, W/4), x: (B, 256, H/8, W/8)

        # ---------- ② Transformer Bottleneck ----------
        # CNN 特征图 → token
        x = self.patch_embed(x)              # (B, N, d_model)
        # + 位置编码
        x = x + self.pos_embed
        # 12 层全局自注意力
        for layer in self.transformer_layers:
            x = layer(x)                    # (B, N, d_model)

        # token 序列 → 特征图
        h = H // 8
        w = W // 8
        x = x.transpose(1, 2)              # (B, d_model, N)
        x = x.view(B, -1, h, w)            # (B, d_model, h, w)
        # 投影回 CNN 通道数
        x = self.decoder_proj(x)            # (B, 256, h, w)

        # ---------- ③ CNN Decoder（跳跃连接）----------
        x = self.dec3(enc3, x)              # cat(enc3, up(x)) → (B, 128, H/4, W/4)
        x = self.dec2(enc2, x)              # cat(enc2, up(x)) → (B, 64,  H/2, W/2)
        x = self.dec1(enc1, x)              # cat(enc1, up(x)) → (B, 64,  H,   W)

        # ---------- ④ Head ----------
        return self.head(x)                 # (B, n_classes, H, W)

## 5. TransUNet vs U-Net vs ViT 对照表

| 组件 | ViT | U-Net | TransUNet |
|---|---|---|---|
| 输入层 | PatchEmbedding（切原图） | DoubleConv（3×3 卷积） | DoubleConv（3×3 卷积） |
| 下采样 | 无（全局注意力） | MaxPool（4 级） | MaxPool（3 级） |
| 深层处理 | Transformer Encoder | DoubleConv | **Transformer Encoder** |
| 上采样 | 无 | ConvTranspose2d + skip | ConvTranspose2d + skip |
| CLS token | ✓ | ✗ | ✗ |
| Skip Connection | ✗ | ✓（4 级） | ✓（3 级） |
| 用在哪 | 分类 | 分割 | **分割（需要全局+局部）** |
| 瓶颈感受野 | 全局 | 3×3 局部 | **全局** |